<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/Optimisation_2_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#!pip install qiskit
#!pip install qiskit-aer
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT, QFTGate # Import QFTGate
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
import numpy as np
from math import gcd
from sympy.ntheory import factorint
import random

def optimized_shor_factor_15():
    """
    Optimized Shor's algorithm with adaptive 'a' selection.
    Pre-selects bases with known favorable properties.
    """
    N = 15

    # Optimization 1: Pre-compute optimal bases using classical preprocessing

    # For N=15, we know the orders: (even, good)
    # a=2 → r=4
    # a=4 → r=2
    # a=7 → r=4
    # a=8 → r=4
    # a=11 → r=2
    # a=13 → r=4
    # a=14 → r=2

    # Rank bases by probability of success
    optimal_bases = {
        7: {'r': 4, 'success_prob': 0.95},
        4: {'r': 2, 'success_prob': 0.90},
        2: {'r': 4, 'success_prob': 0.85},
        8: {'r': 4, 'success_prob': 0.85},
        11: {'r': 2, 'success_prob': 0.90},
        13: {'r': 4, 'success_prob': 0.85},
        14: {'r': 2, 'success_prob': 0.90}
    }

    # Trys best bases first
    for a in sorted(optimal_bases.keys(),
                    key=lambda x: optimal_bases[x]['success_prob'],
                    reverse=True):
        print(f"\nTrying a = {a} (success probability: {optimal_bases[a]['success_prob']})")

        # Optimization 2: Use fewer shots for high-probability bases
        shots = 1024 if optimal_bases[a]['success_prob'] > 0.9 else 2048

        result = shor_with_specific_a(N, a, shots)
        if result:
            print(f"✓ Factor found with a={a}")
            return result

    # Fallback to random selection
    return shor_random_a(N)

def shor_with_specific_a(N, a, shots):
    """Execute Shor's algorithm with a specific a"""
    # Original line: n_count = N.bit_length() * 2
    # Using the value from the example's global scope for N=15
    n_count = 3  # 3 qubits for phase estimation

    # Creates circuit
    qc = QuantumCircuit(n_count + 4, n_count)  # 3 counting qubits + 4 work qubits

    # Initialize work register to |1>
    qc.x(n_count)  # qubit n_count = |1>

    # Apply Hadamard to counting qubits
    for i in range(n_count):
        qc.h(i)

    # Controlled modular exponentiation for a^x mod 15
    # For a=7 and N=15, the period r=4
    # Implementation of controlled U^(2^k) gates

    # U = multiplication by 7 mod 15
    # U|1> = |7>, U^2|1> = |4>, U^4|1> = |1>

    # Apply controlled gates (simplified version for demonstration)
    # Control qubit 0 (2^0)
    qc.cx(0, n_count)      # controlled swap for demonstration
    qc.cx(0, n_count+1)

    # Control qubit 1 (2^1)
    qc.cx(1, n_count)
    qc.cx(1, n_count+2)

    # Control qubit 2 (2^2)
    qc.cx(2, n_count)
    qc.cx(2, n_count+3)

    # Apply inverse QFT
    # Old: qft_circ = QFT(n_count, inverse=True, do_swaps=True)
    qft_circ = QFTGate(num_qubits=n_count).inverse()
    qc.append(qft_circ, range(n_count))

    # Measure counting qubits
    qc.measure(range(n_count), range(n_count))

    print("Shor's algorithm circuit for N=15:")
    print(qc)

    # Execute
    backend = Aer.get_backend('qasm_simulator')
    qc_transpiled = transpile(qc, backend)
    job = backend.run(qc_transpiled, shots=shots)
    result = job.result()
    counts = result.get_counts()

    print("\nMeasurement results:")
    print(counts)

    # Processes results
    measured_phases = []
    for output, count_val in counts.items():
        # Convert binary to decimal
        phase = int(output, 2) / (2**n_count)
        measured_phases.append((phase, count_val))

    # Finds period using continued fractions
    possible_r = []
    for phase, count_val in measured_phases:
        if phase == 0:
            continue
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator
        if r not in [x[0] for x in possible_r] and r > 0:
            possible_r.append((r, count_val))

    # Sorts by frequency
    possible_r.sort(key=lambda x: x[1], reverse=True)

    # Trys each candidate r
    for r, freq in possible_r:
        print(f"\nTesting r = {r} (frequency: {freq})")
        if r % 2 == 0:
            # Check if we found a factor
            candidate1 = gcd(a**(r//2) - 1, N)
            candidate2 = gcd(a**(r//2) + 1, N)
            print(f"  Candidates: {candidate1}, {candidate2}")
            if candidate1 not in [1, N] and candidate1 > 1:
                print(f"\n✓ Factor found: {candidate1}")
                return candidate1
            if candidate2 not in [1, N] and candidate2 > 1:
                print(f"\n✓ Factor found: {candidate2}")
                return candidate2

    print("\n✗ No factor found. Try a different value for 'a'.")
    return None

# A placeholder for shor_random_a if it's called by optimized_shor_factor_15
def shor_random_a(N):
    # For demonstration, a known factor of N=15 will be used
    print(f"\nFalling back to random 'a' selection for N={N}. (Placeholder)")
    # In a real scenario, this would involve selecting a random 'a' and running Shor's
    return None # Or return a factor like 3 or 5 for N=15 if successful

    print("Executing Shor's algorithm for factoring 15...")
factor = optimized_shor_factor_15()
if factor:
    print(f"Final factor found: {factor}")
else:
    print("Could not find a factor for N=15.")


Trying a = 7 (success probability: 0.95)
Shor's algorithm circuit for N=15:
     ┌───┐                              ┌─────────┐┌─┐      
q_0: ┤ H ├──■────■──────────────────────┤0        ├┤M├──────
     ├───┤  │    │                      │         │└╥┘┌─┐   
q_1: ┤ H ├──┼────┼────■────■────────────┤1 qft_dg ├─╫─┤M├───
     ├───┤  │    │    │    │            │         │ ║ └╥┘┌─┐
q_2: ┤ H ├──┼────┼────┼────┼────■────■──┤2        ├─╫──╫─┤M├
     ├───┤┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │  └─────────┘ ║  ║ └╥┘
q_3: ┤ X ├┤ X ├──┼──┤ X ├──┼──┤ X ├──┼──────────────╫──╫──╫─
     └───┘└───┘┌─┴─┐└───┘  │  └───┘  │              ║  ║  ║ 
q_4: ──────────┤ X ├───────┼─────────┼──────────────╫──╫──╫─
               └───┘     ┌─┴─┐       │              ║  ║  ║ 
q_5: ────────────────────┤ X ├───────┼──────────────╫──╫──╫─
                         └───┘     ┌─┴─┐            ║  ║  ║ 
q_6: ──────────────────────────────┤ X ├────────────╫──╫──╫─
                                   └───┘            ║  ║  ║ 
c: 3/═══